      ## Pipelining the conversion of points from a neuroglancer to points within a CCF region

### Steps:
    1. Get points from a neuroglancer link with mutable annotation layer
    2. register points to CCF
    3. Identify points that are within provided CCF region
    4. Run a signed distance algorithm to include nearby cells based on given threshold
    
### Outputs:
    1. CSV file containing the coordinates of all points in imaging space with labeled as inside, boardering, 
       and outside of mesh
    2. All points in CCF space without any labels
    3. CSV file containing the coordinates of all points in CCF space with labeled as inside, boardering, 
       and outside of mesh

In [1]:
# parameters for obtaining cells from NG json and needed metadata

params = {
    'data_folder': '../data', #Root folder for all data
    'annotation_folder': 'finalized', #Subfolder with annotation JSONs
    'tp_name': 'PK_LC_all_annotations', #Name of layer in JSON that contains the true positive cells
    'mesh_regions': [771, 997], #ID for the largest region to subset (i.e. Pons) and brain boundary
    'mesh_root': '../code/ccf_files/CCF_meshes/json_verts_float', # location of the mesh files
    'ccf_res': 25, # Resolution of mesh in ccf space
    'template_res': [14.4, 14.4, 16], # resolution of LS template
    'resolution': 5_000, #resampling mesh resolution
    'threshold': 300, # boudary threshold for signed distance
    'ccf_template': '../data/lightsheet_template_ccf_registration/ccf_average_template_25.nii.gz',
    'smartspim_template': '../data/lightsheet_template_ccf_registration/smartspim_lca_template_25.nii.gz',
    'ccf_transforms': [
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_1Warp.nii.gz',
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_0GenericAffine.mat'
    ],
    'json_path': '../results/jsons',
    'csv_path': '../results/raw_space'
}

In [7]:
#Collect folder locations and metadata for datasets
import os
from glob import glob
from natsort import natsorted
from utils import json_to_xml, utils

folders = glob(os.path.join(params['data_folder'], 'SmartSPIM_*'))
data = {}
for folder in folders:
    lt_id = folder.split('_')[1]
    
    register_folder =  natsorted(glob(os.path.join(folder, 'image_atlas_alignment', 'Ex_*_Em_*')))[-1]
    template_transforms = [
        f"{register_folder}/ls_to_template_SyN_1Warp.nii.gz",
        f"{register_folder}/ls_to_template_SyN_0GenericAffine.mat"
    ]
    
    res_path = natsorted(glob(os.path.join(folder, 'image_tile_fusing', 'OMEZarr', 'Ex_*_Em_*.zarr/0/.zarray')))[-1]
    input_res = json_to_xml.read_json_as_dict(res_path)["shape"]

    input_dims = [
        input_res[-1],
        input_res[-2],
        input_res[-3],
    ]
    
    try:
        annotation_file = glob(os.path.join(params['data_folder'], params['annotation_folder'], f'{lt_id}*.json'))[0]
        annotation_data = json_to_xml.read_json_as_dict(annotation_file)
    except:
        print(f"Need to add annotation JSON for dataset {lt_id}.")
        continue
    
        
    cells = []
    for layers in annotation_data['layers']:
        if isinstance(layers['source'], dict):
            for cell in layers['annotations']:
                cells.append(
                    [
                        cell['point'][0],
                        cell['point'][1],
                        cell['point'][2]
                    ]
                )
                
    acquisition = json_to_xml.read_json_as_dict(os.path.join(folder, 'acquisition.json'))
    
    data[lt_id] = {
        "template_transforms": template_transforms,
        "annotation_file": annotation_file,
        "orientation": acquisition['axes'],
        "input_dims": input_dims,
        "cell_locations": cells,
        "ng_file": annotation_data,
    }
    
print(f"Will be processing {len(data)} datasets")

Will be processing 37 datasets


In [9]:
np.array(data['762199']['cell_locations'], dtype = np.float32)

array([[1631., 6596., 3559.],
       [1749., 6662., 3567.],
       [1763., 6472., 3555.],
       ...,
       [1968., 6721., 3546.],
       [2052., 6796., 3504.],
       [1959., 6840., 3494.]], dtype=float32)

In [11]:
# transform parent mesh for each dataset and grab only cells within mesh
import os
import vedo
import json

import numpy as np
import pandas as pd
import point_cloud_utils as pcu

from utils import utils
from utils import pipeline_utils as pu


if not os.path.exists(params['json_path']):
    os.mkdir(params['json_path'])
    
if not os.path.exists(params['csv_path']):
    os.mkdir(params['csv_path'])

for lt_id, dataset in data.items():
    
    meshes = {}
    
    for struct in params['mesh_regions']:
    
        verts, faces = pu.load_json_mesh(params['mesh_root'], str(struct))
        verts = verts[:, [0, 2, 1]] / params['ccf_res']
    
        verts_ls, faces_ls = pu.warp_mesh(
            verts,
            faces,
            params['ccf_template'],
            params['smartspim_template'],
            params['ccf_transforms'],
            dataset['template_transforms']
        )
    
        scale = [params['ccf_res']/dim for dim in params['template_res']]
        for i in range(3):
            verts_ls[:, i] = verts_ls[:, i] * scale[i]
            
        verts_ls *= 2**3
        
        orientation = utils.get_orientation(dataset['orientation'])
        _, swapped, mat = utils.get_orientation_transform('ras', orientation)

        verts_orient = pu.orient_mesh(
            vert_list=verts_ls, 
            reg_dims=data[lt_id]['input_dims'],
            ds=1,
            orient='ras',
            orient_matrix=mat,
            institute='AIND',
        )

        verts_orient = verts_orient[:, swapped]
        
        
        if struct != 997:
            
            print(f'Getting cells for {struct} in dataset {lt_id}')

            vw, fw = pcu.make_mesh_watertight(verts_orient, faces_ls, params['resolution'])
            
            pts = np.array(dataset['cell_locations'], dtype = np.double)
            sdf, fid, bc = pcu.signed_distance_to_mesh(pts, vw, fw)
            
            sd_points = []
            for c, s in enumerate(sdf):
                if s < 0:
                    sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'inside'])
                elif s < params['threshold']:
                    sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'boarder'])
                else:
                    sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'outside'])
            
            meshes[struct] = {
                'faces': fw,
                'verts': vw
            }
            
            meshes[struct]['cells'] = sd_points
            
            

            df = pd.DataFrame(sd_points, columns = ['Z', 'Y', 'X', 'Location'])
            df.to_csv(f"{params['csv_path']}/{str(lt_id)}_cells.csv")
            
            #might want to add later, but not needed now
            #new_ng_link = add_annotation_layer(dataset['ng_file'], locations)
            #save_json(new_ng_link, f"{params['json_path']}/{str(lt_id)}_new.json")
        else:
            meshes[struct] = {
                'faces': faces_ls,
                'verts': verts_orient
            }
        
    data[lt_id]['meshes'] = meshes

Getting cells for 771 in dataset 796809
Getting cells for 771 in dataset 746046
Getting cells for 771 in dataset 746045
Getting cells for 771 in dataset 731907
Getting cells for 771 in dataset 755073
Getting cells for 771 in dataset 804461
Getting cells for 771 in dataset 801624
Getting cells for 771 in dataset 804462
Getting cells for 771 in dataset 762199
Getting cells for 771 in dataset 751017
Getting cells for 771 in dataset 784354
Getting cells for 771 in dataset 751035
Getting cells for 771 in dataset 755069
Getting cells for 771 in dataset 791275
Getting cells for 771 in dataset 725231
Getting cells for 771 in dataset 791270
Getting cells for 771 in dataset 751023
Getting cells for 771 in dataset 751024
Getting cells for 771 in dataset 784361
Getting cells for 771 in dataset 791272
Getting cells for 771 in dataset 744119
Getting cells for 771 in dataset 755072
Getting cells for 771 in dataset 793600
Getting cells for 771 in dataset 744117
Getting cells for 771 in dataset 793594


In [4]:
# Plot example
import k3d

plot = k3d.plot()

dataset = "804462"

v, f = data[dataset]['meshes'][997]['verts'], data[dataset]['meshes'][997]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh

vw, fw = data[dataset]['meshes'][771]['verts'], data[dataset]['meshes'][771]['faces']
mesh = k3d.mesh(vw, fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

cells = data[dataset]['meshes'][771]['cells']
df = pd.DataFrame(cells, columns = ['Z', 'Y', 'X', 'Location'])

inside = df.loc[df['Location'] == "inside", ["Z", "Y", "X"]].values
boarder = df.loc[df['Location'] == "boarder", ["Z", "Y", "X"]].values
outside = df.loc[df['Location'] == "outside", ["Z", "Y", "X"]].values

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()




/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [6]:
# Plot example
plot = k3d.plot()

dataset = "801624"

v, f = data[dataset]['meshes'][997]['verts'], data[dataset]['meshes'][997]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh

vw, fw = data[dataset]['meshes'][771]['verts'], data[dataset]['meshes'][771]['faces']
mesh = k3d.mesh(vw, fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

cells = data[dataset]['meshes'][771]['cells']
df = pd.DataFrame(cells, columns = ['Z', 'Y', 'X', 'Location'])

inside = df.loc[df['Location'] == "inside", ["Z", "Y", "X"]].values
boarder = df.loc[df['Location'] == "boarder", ["Z", "Y", "X"]].values
outside = df.loc[df['Location'] == "outside", ["Z", "Y", "X"]].values

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [ ]:
# Plot example
plot = k3d.plot()

dataset = "755073"

v, f = data[dataset]['meshes'][997]['verts'], data[dataset]['meshes'][997]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh

vw, fw = data[dataset]['meshes'][771]['verts'], data[dataset]['meshes'][771]['faces']
mesh = k3d.mesh(vw, fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

cells = data[dataset]['meshes'][771]['cells']
df = pd.DataFrame(cells, columns = ['Z', 'Y', 'X', 'Location'])

inside = df.loc[df['Location'] == "inside", ["Z", "Y", "X"]].values
boarder = df.loc[df['Location'] == "boarder", ["Z", "Y", "X"]].values
outside = df.loc[df['Location'] == "outside", ["Z", "Y", "X"]].values

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

In [ ]:
# Plot example
plot = k3d.plot()

dataset = "762196"

v, f = data[dataset]['meshes'][997]['verts'], data[dataset]['meshes'][997]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh

vw, fw = data[dataset]['meshes'][771]['verts'], data[dataset]['meshes'][771]['faces']
mesh = k3d.mesh(vw, fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

cells = data[dataset]['meshes'][771]['cells']
df = pd.DataFrame(cells, columns = ['Z', 'Y', 'X', 'Location'])

inside = df.loc[df['Location'] == "inside", ["Z", "Y", "X"]].values
boarder = df.loc[df['Location'] == "boarder", ["Z", "Y", "X"]].values
outside = df.loc[df['Location'] == "outside", ["Z", "Y", "X"]].values

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 25, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

In [12]:
# Parameters for Registering points
# set parameters
from glob import glob

reg_params = {
    'csv_files': glob('../results/raw_space/*.csv'), #csv files in raw image space to be transformed
    'mesh_path': '../code/ccf_files/CCF_meshes/json_verts_float', #location of ccf meshes for plotting
    'save_path': '../results/registered',
    "smartspim_template": '../data/lightsheet_template_ccf_registration/smartspim_lca_template_25.nii.gz', # location of smartspim template
    "ccf_template": '../data/lightsheet_template_ccf_registration/ccf_average_template_25.nii.gz', #location of ccf template
    'template_to_ccf': [
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_0GenericAffine.mat',
        '../data/lightsheet_template_ccf_registration/syn_1InverseWarp.nii.gz'
    ], #transforms for ls_tempalte -> ccf 
    'downsample': 3, #the level used for registering the raw image to ls_template
    'ccf_res': 25,
    'scaling': [16 / 25, 14.4 / 25, 14.4 / 25], #resolution change from downsampled image to ants object
    'institute': 'AIND' #institute that imaged brain
}

In [13]:
# loop over csv files and register
import os
import re
import numpy as np
from utils import utils


registered_cells = {}

for file in reg_params['csv_files']:
    
    lt_id = re.findall('[0-9]{6}', file)[0]
    print(f'registering dataset: {lt_id}')
    
    acquisition_file = glob(f'../data/SmartSPIM_{lt_id}*/acquisition.json')[0]
    zarr_path = glob(f'../data/SmartSPIM_{lt_id}*/image_tile_fusing/OMEZarr/*.zarr')[0]
    
    
    print(f'location of acquisition file: {acquisition_file}\nlocation of zarr: {zarr_path}')
    
    ls_to_template = [
        glob(f'../data/SmartSPIM_{lt_id}*/image_atlas_alignment/**/ls_to_template_SyN_0GenericAffine.mat')[0],
        glob(f'../data/SmartSPIM_{lt_id}*/image_atlas_alignment/**/ls_to_template_SyN_1InverseWarp.nii.gz')[0]
    ]
    
    print(f'lightsheet to template transforms: {ls_to_template}')
    
    res_metadata = f"{zarr_path}/0/.zarray"
    
    input_res = utils.read_json_as_dict(res_metadata)["shape"]

    # input res is returned in order tczyx, here we use xzy
    # where z is the imaging axis
    res = [
        input_res[-3],
        input_res[-2],
        input_res[-1],
    ]

    # get dimensions at registered scale
    ds = 2**reg_params['downsample']
    reg_dims = [dim / ds for dim in res]
    
    # get orientation information
    orientation = utils.read_json_as_dict(acquisition_file)
    orient = utils.get_orientation(orientation['axes'])
    template_params = utils.get_template_info(reg_params["smartspim_template"])
    
    _, swapped, mat = utils.get_orientation_transform(
        orient, 
        template_params["orientation"]
    )
    
    raw_cells = utils.read_cells_from_csv(
        cell_likelihoods_path=file,
        reg_dims=reg_dims,
        ds=ds,
        orient=orient,
        orient_matrix=mat,
        institute=reg_params['institute'],
    )
    
    scaled_cells = utils.scale_cells(raw_cells, reg_params['scaling'])
    scaled_cells = np.array(scaled_cells)
    orient_cells = scaled_cells[:, swapped]


    #Converting oriented cells into ANTs physical space
    template_params = utils.get_template_info(reg_params["smartspim_template"])
    ants_cells = utils.convert_to_ants_space(template_params, orient_cells)
    
    #Registering Cells to SmartSPIM template
    template_cells = utils.apply_transforms_to_points(
        ants_cells, ls_to_template, invert=(True, False)
    )

    #Convert template cells into CCF space and orientation
    ccf_pts = utils.apply_transforms_to_points(
        template_cells, reg_params['template_to_ccf'], invert=(True, False)
    )

    #Convert cells back into index space
    ccf_params = utils.get_template_info(reg_params["ccf_template"])
    ccf_cells = utils.convert_from_ants_space(ccf_params, ccf_pts)

    _, swapped, _ = utils.get_orientation_transform(
        template_params["orientation"], ccf_params["orientation"]
    )
    
    cells_transformed = ccf_cells[:, swapped]
    
    # Save the transformed coordinates for this lt_id
    pu.save_coordinates_with_indices_to_csv(cells_transformed, lt_id, output_dir = reg_params['save_path'])  # Save to CSV
    
    registered_cells[lt_id] = cells_transformed

registering dataset: 796809
location of acquisition file: ../data/SmartSPIM_796809_2025-08-07_15-59-16_stitched_2025-08-08_20-45-35/acquisition.json
location of zarr: ../data/SmartSPIM_796809_2025-08-07_15-59-16_stitched_2025-08-08_20-45-35/image_tile_fusing/OMEZarr/Ex_561_Em_600.zarr
lightsheet to template transforms: ['../data/SmartSPIM_796809_2025-08-07_15-59-16_stitched_2025-08-08_20-45-35/image_atlas_alignment/Ex_639_Em_680/ls_to_template_SyN_0GenericAffine.mat', '../data/SmartSPIM_796809_2025-08-07_15-59-16_stitched_2025-08-08_20-45-35/image_atlas_alignment/Ex_639_Em_680/ls_to_template_SyN_1InverseWarp.nii.gz']
Saved coordinates and indices for 796809 to ../results/registered/796809_ccf.csv


registering dataset: 746046
location of acquisition file: ../data/SmartSPIM_746046_2024-09-26_22-22-35_stitched_2024-09-28_22-26-50/acquisition.json
location of zarr: ../data/SmartSPIM_746046_2024-09-26_22-22-35_stitched_2024-09-28_22-26-50/image_tile_fusing/OMEZarr/Ex_639_Em_667.zarr
lights

In [14]:
#Create Simplified mesh and run signed distance algorithm on points
from glob import glob

sd_params = {
    'csv_files': glob('../results/registered/*.csv'), #csv files ccf space
    'mesh_path': '../code/ccf_files/CCF_meshes/json_verts_float', #location of ccf meshes for plotting
    'save_path': '../results/final_results',
    'mesh_ids': [771, 997], # CCF region IDs
    'ccf_res': 25, # resolution of the ccf in microns
    'resolution': 5_000, #resolution of the resampeld mesh
    'threshold': 300, #threshold of the signed distance
}

print(sd_params['csv_files'])

['../results/registered/796809_ccf.csv', '../results/registered/746046_ccf.csv', '../results/registered/746045_ccf.csv', '../results/registered/731907_ccf.csv', '../results/registered/755073_ccf.csv', '../results/registered/804461_ccf.csv', '../results/registered/801624_ccf.csv', '../results/registered/804462_ccf.csv', '../results/registered/762199_ccf.csv', '../results/registered/751017_ccf.csv', '../results/registered/784354_ccf.csv', '../results/registered/751035_ccf.csv', '../results/registered/755069_ccf.csv', '../results/registered/791275_ccf.csv', '../results/registered/725231_ccf.csv', '../results/registered/791270_ccf.csv', '../results/registered/751023_ccf.csv', '../results/registered/751024_ccf.csv', '../results/registered/784361_ccf.csv', '../results/registered/791272_ccf.csv', '../results/registered/744119_ccf.csv', '../results/registered/755072_ccf.csv', '../results/registered/793600_ccf.csv', '../results/registered/744117_ccf.csv', '../results/registered/793594_ccf.csv',

In [15]:
# get CCF meshes

mesh_dict = {}

for mesh_id in sd_params['mesh_ids']:
    verts, faces = pu.load_json_mesh(sd_params['mesh_path'], str(mesh_id))
    mesh_dict[str(mesh_id)] = {
        "verts": verts,
        "faces": faces
    }

In [16]:
# create resampled mesh and visualize: 
import k3d
import matplotlib as mpl
import point_cloud_utils as pcu

from utils import pipeline_utils as pu

mesh_id = '771'
v, f = mesh_dict[str(mesh_id)]['verts'], mesh_dict[str(mesh_id)]['faces']
vw, fw = pcu.make_mesh_watertight(v, f, params['resolution'])

plot = k3d.plot()

mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(255,0,255), opacity=0.3)
plot += mesh

mesh = k3d.mesh(vw, fw, color=pu.rgb_to_hex(0,255, 255), opacity=0.3)
plot += mesh

plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [17]:
# get signed distance for each point
import pandas as pd

"""
Need to orient points to match the mesh files


Orientation of CCF Template (points out of registration)

X: AP (Anterior -> Posterior)

Y: DV (Superior -> Inferior)

Z: ML (Left -> Right)


Orientation of mesh

X: ML (Right -> Left)

Y: DV (Superior -> Inferior)

Z: AP (Anterior -> Posterior)
"""

if not os.path.exists(sd_params['save_path']):
    os.mkdir(sd_params['save_path'])
    
ccf_orientation = ccf_params["orientation"]
print(ccf_orientation)
mesh_orientation = 'RSA'

_, swapped, _ = utils.get_orientation_transform(
    ccf_orientation, mesh_orientation
)

print(swapped)

point_dict = {}
for file in sd_params['csv_files']:
    lt_id = os.path.basename(file).split('_')[0]
    df = pd.read_csv(file, index_col = 0)
    pts = df.values
    pts = pts[:, swapped] * sd_params['ccf_res']
    
    vw = np.array(vw, order = 'F')
    sdf, fid, bc = pcu.signed_distance_to_mesh(pts, vw, fw)

    pts /= sd_params['ccf_res']
    
    sd_points = []
    for c, s in enumerate(sdf):
        if s < 0:
            sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'inside'])
        elif s < sd_params['threshold']:
            sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'boarder'])
        else:
            sd_points.append([pts[c, 0], pts[c, 1], pts[c, 2], 'outside'])
    
    points_df = pd.DataFrame(sd_points, columns = ['ML', 'DV', 'AP', 'Location'])
    
    filename = f"{lt_id}_registered_pts.csv"
    points_df.to_csv(f"{sd_params['save_path']}/{filename}")
    
    print(f"Saving data for dataset {lt_id}: {filename}\n")
    

ASL
[2 1 0]
Saving data for dataset 796809: 796809_registered_pts.csv

Saving data for dataset 746046: 746046_registered_pts.csv

Saving data for dataset 746045: 746045_registered_pts.csv

Saving data for dataset 731907: 731907_registered_pts.csv

Saving data for dataset 755073: 755073_registered_pts.csv

Saving data for dataset 804461: 804461_registered_pts.csv

Saving data for dataset 801624: 801624_registered_pts.csv

Saving data for dataset 804462: 804462_registered_pts.csv

Saving data for dataset 762199: 762199_registered_pts.csv

Saving data for dataset 751017: 751017_registered_pts.csv

Saving data for dataset 784354: 784354_registered_pts.csv

Saving data for dataset 751035: 751035_registered_pts.csv

Saving data for dataset 755069: 755069_registered_pts.csv

Saving data for dataset 791275: 791275_registered_pts.csv

Saving data for dataset 725231: 725231_registered_pts.csv

Saving data for dataset 791270: 791270_registered_pts.csv

Saving data for dataset 751023: 751023_regis

In [18]:
#Plot example
plot = k3d.plot()
dataset = '793594'
df = pd.read_csv(f"{sd_params['save_path']}/{dataset}_registered_pts.csv", index_col = 0)

inside = df.loc[df['Location'] == "inside", ["ML", "DV", "AP"]].values
boarder = df.loc[df['Location'] == "boarder", ["ML", "DV", "AP"]].values
outside = df.loc[df['Location'] == "outside", ["ML", "DV", "AP"]].values

        
v, f = mesh_dict[str(997)]['verts'] / sd_params['ccf_res'], mesh_dict[str(997)]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh
               
mesh = k3d.mesh(vw / sd_params['ccf_res'], fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [14]:
#Plot example
plot = k3d.plot()
dataset = '801624'
df = pd.read_csv(f"{sd_params['save_path']}/{dataset}_registered_pts.csv", index_col = 0)

inside = df.loc[df['Location'] == "inside", ["ML", "DV", "AP"]].values
boarder = df.loc[df['Location'] == "boarder", ["ML", "DV", "AP"]].values
outside = df.loc[df['Location'] == "outside", ["ML", "DV", "AP"]].values
        
v, f = mesh_dict[str(997)]['verts'] / sd_params['ccf_res'], mesh_dict[str(997)]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh
               
mesh = k3d.mesh(vw / sd_params['ccf_res'], fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [14]:
#Plot example
plot = k3d.plot()
dataset = '796810'
df = pd.read_csv(f"{sd_params['save_path']}/{dataset}_registered_pts.csv", index_col = 0)

inside = df.loc[df['Location'] == "inside", ["ML", "DV", "AP"]].values
boarder = df.loc[df['Location'] == "boarder", ["ML", "DV", "AP"]].values
outside = df.loc[df['Location'] == "outside", ["ML", "DV", "AP"]].values
        
v, f = mesh_dict[str(997)]['verts'] / sd_params['ccf_res'], mesh_dict[str(997)]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh
               
mesh = k3d.mesh(vw / sd_params['ccf_res'], fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [ ]:
#Plot example
plot = k3d.plot()
dataset = '762196'
df = pd.read_csv(f"{sd_params['save_path']}/{dataset}_registered_pts.csv", index_col = 0)

inside = df.loc[df['Location'] == "inside", ["ML", "DV", "AP"]].values
boarder = df.loc[df['Location'] == "boarder", ["ML", "DV", "AP"]].values
outside = df.loc[df['Location'] == "outside", ["ML", "DV", "AP"]].values
        
v, f = mesh_dict[str(997)]['verts'] / sd_params['ccf_res'], mesh_dict[str(997)]['faces']
mesh = k3d.mesh(v, f, color=pu.rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh
               
mesh = k3d.mesh(vw / sd_params['ccf_res'], fw, color=pu.rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 2, color = pu.rgb_to_hex(0, 0, 0))
plot += pts

plot.display()